# Transformer Decoder-Only モデルを使用したテキスト生成

## 概要

このサンプルでは、KerasNLP（現 Keras Hub）を使用して、Transformer Decoder を用いた小規模な生成モデルを構築します。生成モデルを使用すると、プロンプトから高度なテキストを生成できます。このラボでは、Transformer スタックのデコーダーのみを使用するモデルを構築します。各ステージにおいて、特定の単語に対するアテンション レイヤは、文中のその単語より前に配置された単語にのみアクセスできます。これらのモデルは、しばしば自己回帰（auto-regressive）モデルと呼ばれます。デコーダーのみのモデルの例としては、[Transformer XL](https://arxiv.org/pdf/1901.02860.pdf)、[GPT-2](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf)、[CTRL](https://arxiv.org/pdf/1909.05858.pdf) などがあります。

複数の小説から作成されたデータセットである [simplebooks-92](https://arxiv.org/abs/1911.12391) コーパスでモデルをトレーニングします。このデータセットは 1,573 冊の Gutenberg 書籍から作成されました。語彙数が少なく単語頻度が高いため、パラメータ数の少ない生成モデルをトレーニングする際に有利であり、このサンプルに適したデータセットです。

このノートブックでは、KerasNLP のトークン化、レイヤ、メトリクスを使用してトレーニング プロセスを簡素化する方法を示し、KerasNLP のサンプリング ユーティリティを使用して出力テキストを生成する方法を説明します。

### 学習目標
- WordPiece トークナイザーを使用して生成モデル用のデータセットを準備・トレーニングする方法を学ぶ
- Keras NLP を使用して生成モデルを構築する方法を学ぶ
- プロンプトからテキストを出力するためのさまざまな推論手法を学ぶ

## セットアップ

このノートブックを実行するには、`keras_hub` が必要です。Keras Hub は、シンプル、柔軟、かつ高速を目指した事前トレーニング済みモデル ライブラリです。TensorFlow、JAX、または PyTorch とネイティブに連携します。Keras NLP は、このノートブックで生成モデルを構築するのに非常に役立つ Transformer レイヤを提供します。

In [ ]:
import os
import warnings

warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

In [ ]:
import keras
import keras_hub
import tensorflow as tf

### 始める前に
トレーニングに時間がかかりすぎないように、ノートブック インスタンスに GPU（1 x NVIDIA Tesla T4 で十分です）が接続されていることを確認してください。

GPU が接続されているかどうかを確認するには、次のコマンドを実行します:

In [ ]:
# 1 つの GPU が接続されている場合、これは "Num GPUs Available: 1" と出力されるはずです
print("Num GPUs Available: ", len(tf.config.list_physical_devices("GPU")))

## 設定とハイパーパラメータ

ここでいくつかのハイパーパラメータを初期化します。ここでの `SEQ_LEN` はデータセット内の文の最大長を定義し、これを `WordPieceTokenizer` の長さとして使用します。また、短すぎる文をデータセットから除外してクリーンアップするために `MIN_TRAINING_SEQ_LEN` も定義します。

In [ ]:
# データ
BATCH_SIZE = 64
SEQ_LEN = 128
MIN_TRAINING_SEQ_LEN = 450

# モデル
EMBED_DIM = 256
FEED_FORWARD_DIM = 256
NUM_HEADS = 3
NUM_LAYERS = 2
VOCAB_SIZE = 5000  # モデルのパラメータ数を制限

## データの読み込み

このノートブックでは SimpleBooks データセットを使用します。SimpleBooks データセットは、1,573 冊の Gutenberg 書籍で構成され、単語レベルのトークンに対する語彙サイズの比率が小さいのが特徴です。語彙サイズは約 98,000 です。このサイズにより、小型の Transformer モデルに適合させやすくなります。このブロックでは、[TensorFlow のデータ API](https://www.tensorflow.org/api_docs/python/tf/data/TextLineDataset) を使用してデータを読み込みます。 

In [ ]:
file = keras.utils.get_file(
    origin="https://storage.googleapis.com/asl-public/text/data/simplebooks.zip",
    extract=True,
)
data_dir = os.path.expanduser("~/.keras/datasets/simplebooks.zip/simplebooks/")

# simplebooks-92 のトレーニング セットを読み込み、MIN_TRAINING_SEQ_LEN を使用して短い行を除外
raw_train_ds = (
    tf.data.TextLineDataset(data_dir + "simplebooks-92-raw/train.txt")
    .filter(lambda x: tf.strings.length(x) > MIN_TRAINING_SEQ_LEN)
    .batch(BATCH_SIZE)
    .shuffle(buffer_size=256)
)

# simplebooks-92 の検証セットを読み込み、MIN_TRAINING_SEQ_LEN を使用して短い行を除外
raw_val_ds = (
    tf.data.TextLineDataset(data_dir + "simplebooks-92-raw/valid.txt")
    .filter(lambda x: tf.strings.length(x) > MIN_TRAINING_SEQ_LEN)
    .batch(BATCH_SIZE)
)

## トークナイザーのトレーニング

トレーニング データセットから、チューニングされたハイパーパラメータである語彙サイズ `VOCAB_SIZE` を指定して、Keras NLP の [compute_word_piece_vocabulary](https://keras.io/api/keras_hub/tokenizers/compute_word_piece_vocabulary/) を使用してトークナイザーをトレーニングします。モデルのパラメータ数に大きな影響を与えるため、語彙はできるだけ制限したいと考えます。ただし、単語数が*少なすぎる*と語彙外（OOV）サブワードが多すぎるため、適切なバランスが必要です。さらに、語彙内には 3 つのトークンが予約されています。

- シーケンスを `SEQ_LEN` までパディングするための `"[PAD]"`。`WordPieceTokenizer`（およびその他のレイヤ）はデフォルトのパディングとして `0`/`vocab[0]` を考慮するため、このトークンは `reserved_tokens` と `vocab` の両方でインデックス 0 を持ちます。
- OOV サブワード用の `"[UNK]"`。これは `WordPieceTokenizer` のデフォルトの `oov_token="[UNK]"` と一致する必要があります。
- 文の先頭（beginning of sentence）を表す `"[BOS]"`。ただし、ここでは技術的にはトレーニング データの各行の先頭を表すトークンです。

このセルはデータセット全体で WordPiece 語彙を計算するため、実行に 5〜10 分程度かかります。

In [ ]:
# トークナイザーの語彙をトレーニング
print("Training the word piece tokenizer. This will take 5-10 mins...")
vocab = keras_hub.tokenizers.compute_word_piece_vocabulary(
    raw_train_ds,
    vocabulary_size=VOCAB_SIZE,
    lowercase=True,
    reserved_tokens=["[PAD]", "[UNK]", "[BOS]"],
)
print("Training is complete!!")

## トークナイザーの読み込み

語彙データを使用して [keras_hub.tokenizers.WordPieceTokenizer](https://keras.io/api/keras_hub/tokenizers/word_piece_tokenizer/) を初期化します。WordPieceTokenizer は、BERT やその他のモデルで使用されている WordPiece アルゴリズムの効率的な実装です。空白の削除、小文字化、その他の元に戻せない前処理操作を実行します。語彙と入力文が与えられると、WordPiece トークナイザーは文を ID の配列に変換し、定義された `SEQ_LEN` まで文をパディングします。たとえば、

```
vocab = ["[UNK]", "the", "qu", "##ick", "br", "##own", "fox", "."]
inputs = "The quick brown fox."
SEQ_LEN = 10
```

これを `WordPieceTokenizer` に渡すと、次のように返されます。
```
array([1, 2, 3, 4, 5, 6, 7,0,0,0], dtype=int32)
```

In [ ]:
tokenizer = keras_hub.tokenizers.WordPieceTokenizer(
    vocabulary=vocab,
    sequence_length=SEQ_LEN,
    lowercase=True,
)

## データのトークン化

データセットをトークン化し、`features`（特徴量）と `labels`（ラベル）に分割して前処理します。これは言語モデリング タスクであるため、目標は「前の単語」の「特徴量シーケンス」から「次の単語」の「ラベル シーケンス」を予測することです。「特徴量」を取得するために、`[BOS]` トークンを使用して元の文を右にシフトします。

In [ ]:
# packer が開始トークンを追加
start_packer = keras_hub.layers.StartEndPacker(
    sequence_length=SEQ_LEN,
    start_value=tokenizer.token_to_id("[BOS]"),
)


def preprocess(inputs):
    outputs = tokenizer(inputs)
    features = start_packer(outputs)
    labels = outputs
    return features, labels


# トークン化し、トレーニング シーケンスとラベル シーケンスに分割
train_ds = raw_train_ds.map(
    preprocess, num_parallel_calls=tf.data.AUTOTUNE
).prefetch(tf.data.AUTOTUNE)
val_ds = raw_val_ds.map(
    preprocess, num_parallel_calls=tf.data.AUTOTUNE
).prefetch(tf.data.AUTOTUNE)

## モデルの構築

以下のレイヤを使用して、小規模な Transformer-Decoder ベースのテキスト生成モデルを作成します。

- トークンの埋め込みとその位置の埋め込みを組み合わせる 1 つの `keras_hub.layers.TokenAndPositionEmbedding` レイヤ。固定の正弦波埋め込みの代わりにトレーニング可能な位置埋め込みを作成するため、従来の埋め込みレイヤとは異なります。
- ループを使用して作成された複数の `keras_hub.layers.TransformerDecoder` レイヤ。
- 最終的な 1 つの Dense 線形レイヤ。

**注:** このレイヤを構成するさまざまなコンポーネントを確認するには、このレイヤの [ソースコード](https://github.com/keras-team/keras-hub/blob/v0.23.0/keras_hub/src/layers/modeling/token_and_position_embedding.py#L11) を参照してください。

In [ ]:
inputs = keras.layers.Input(shape=(None,), dtype="int32")
# 埋め込みレイヤ
embedding_layer = keras_hub.layers.TokenAndPositionEmbedding(
    vocabulary_size=VOCAB_SIZE,
    sequence_length=SEQ_LEN,
    embedding_dim=EMBED_DIM,
    mask_zero=True,
)
x = embedding_layer(inputs)
# Transformer デコーダー レイヤ
for _ in range(NUM_LAYERS):
    decoder_layer = keras_hub.layers.TransformerDecoder(
        num_heads=NUM_HEADS,
        intermediate_dim=FEED_FORWARD_DIM,
    )
    x = decoder_layer(x)  # 1 つの引数のみを渡すとクロスアテンションがスキップされます
# 出力レイヤ
outputs = keras.layers.Dense(VOCAB_SIZE)(x)
model = keras.Model(inputs=inputs, outputs=outputs)

# 損失メトリクスを設定
loss_fn = keras.losses.SparseCategoricalCrossentropy(from_logits=True)
perplexity = keras_hub.metrics.Perplexity(from_logits=True, mask_token_id=0)

# モデルをコンパイル
model.compile(optimizer="adam", loss=loss_fn, metrics=[perplexity])

モデルのサマリーを見てみましょう。パラメータの大部分は `token_and_position_embedding` と出力の `dense` レイヤに存在しています！
これは、語彙サイズ（`VOCAB_SIZE`）がモデルのサイズに大きな影響を与える一方で、Transformer デコーダー レイヤの数（`NUM_LAYERS`）はそれほど影響しないことを意味します。

In [ ]:
model.summary()

## トレーニング

モデルが準備できたので、`fit()` メソッドを使用してトレーニングしましょう。

In [ ]:
EPOCHS = 1  # より良い結果を得るにはエポック数を増やしてください
print("Training started, this could take 4-10 mins per epoch with a T4 GPU...")
model.fit(train_ds, validation_data=val_ds, verbose=2, epochs=EPOCHS)
print("Training is complete!!")

## 推論

トレーニングされたモデルを使用して、テストを行いパフォーマンスを測定できます。これを行うには、`"[BOS]"` トークンで始まる入力シーケンスをモデルのシードとして与え、ループ内で後続の各トークンの予測を行いながら、モデルから段階的にサンプリングします。

まず、モデル入力と同じ形状を持ち、`"[BOS]"` トークンのみを含むプロンプトを作成しましょう。

In [ ]:
# "packer" レイヤが [BOS] トークンを追加してくれます。
prompt_tokens = start_packer(tokenizer([""]))
prompt_tokens

推論には `keras_hub.samplers` モジュールを使用します。これには、先ほどトレーニングしたモデルをラップするコールバック関数が必要です。このラッパーはモデルを呼び出し、現在生成しているトークンのロジット予測を返します。

**注:** コールバックを定義する際に利用できる、より高度な機能が 2 つあります。1 つ目は、前の生成ステップで計算された状態の `cache` を受け取る機能で、これを使用すると生成を高速化できます。2 つ目は、生成された各トークンの最終的な Dense「隠れ状態」を出力する機能です。これは、繰り返される隠れ状態にペナルティを与えることで繰り返しを回避する `keras_hub.samplers.ContrastiveSampler` によって使用されます。どちらもオプションであるため、ここでは無視します。

In [ ]:
def next(prompt, cache, index):
    logits = model(prompt)[:, index - 1, :]
    # 今のところ隠れ状態は無視します。コントラスト探索にのみ必要です。
    hidden_states = None
    return logits, hidden_states, cache

ラッパー関数の作成は、これらの関数を使用する上で最も複雑な部分です。作成が完了したので、greedy search から順に、さまざまなユーティリティをテストしてみましょう。

### Greedy search（貪欲探索）

各タイムステップで最も確率の高いトークンを貪欲に選択します。言い換えると、モデル出力の argmax を取得します。

In [ ]:
sampler = keras_hub.samplers.GreedySampler()
output_tokens = sampler(
    next=next,
    prompt=prompt_tokens,
    index=1,  # [BOS] トークンの直後からサンプリングを開始。
)
txt = tokenizer.detokenize(output_tokens)
print(f"Greedy search generated text: \n{txt}\n")

ご覧のとおり、greedy search は最初は筋が通っていますが、すぐに同じ言葉を繰り返し始めます。これはテキスト生成における一般的な問題であり、後で説明するいくつかの確率的テキスト生成ユーティリティによって解決できます！

### Beam search（ビーム探索）

大まかに言えば、beam search は各タイムステップで `num_beams` 個の最も確率の高いシーケンスを追跡し、すべてのシーケンスから最適な次のトークンを予測します。より多くの可能性を保持するため、greedy search よりも改善されています。ただし、複数の潜在的なシーケンスを計算して保持する必要があるため、greedy search よりも効率は低くなります。

**注:** `num_beams=1` の beam search は greedy search と同一です。

In [ ]:
sampler = keras_hub.samplers.BeamSampler(num_beams=10)
output_tokens = sampler(
    next=next,
    prompt=prompt_tokens,
    index=1,
)
txt = tokenizer.detokenize(output_tokens)
print(f"Beam search generated text: \n{txt}\n")

greedy search と同様に、beam search も決定論的な手法であるため、すぐに同じ言葉を繰り返し始めます。

### Random search（ランダム探索）

Random search は最初の確率的手法です。各タイムステップで、モデルによって提供された softmax 確率を使用して次のトークンをサンプリングします。

In [ ]:
sampler = keras_hub.samplers.RandomSampler()
output_tokens = sampler(
    next=next,
    prompt=prompt_tokens,
    index=1,
)
txt = tokenizer.detokenize(output_tokens)
print(f"Random search generated text: \n{txt}\n")

これで繰り返しがなくなりました！ただし、random search では、語彙内の任意の単語がこのサンプリング手法で出現する可能性があるため、意味の通らない単語が表示されることがあります。これは、次の探索ユーティリティである top-k search で解決されます。

### Top-K search（Top-K 探索）

random search と同様に、モデルによって提供された確率分布から次のトークンをサンプリングします。唯一の違いは、ここでは最も確率の高い上位 `k` 個のトークンを選択し、サンプリングする前にそれらのトークン全体に確率質量を分散させることです。こうすることで、確率の低いトークンからサンプリングすることがなくなり、意味不明な単語が少なくなります！

In [ ]:
sampler = keras_hub.samplers.TopKSampler(k=10)
output_tokens = sampler(
    next=next,
    prompt=prompt_tokens,
    index=1,
)
txt = tokenizer.detokenize(output_tokens)
print(f"Top-K search generated text: \n{txt}\n")

### Top-P search（Top-P 探索）

top-k search でも改善の余地があります。top-k search では、数値 `k` が固定されているため、どのような確率分布に対しても同じ数のトークンが選択されます。確率質量が 2 つの単語に集中しているシナリオと、10 個の単語に均等に分散しているシナリオの 2 つを考えてみましょう。`k=2` を選択すべきでしょうか、それとも `k=10` を選択すべきでしょうか？あらゆるケースに万能な `k` は存在しません。

そこで登場するのが top-p search です！`k` を選択する代わりに、上位トークンの確率の合計が達してほしい確率 `p` を選択します。これにより、確率分布に基づいて `k` を動的に調整できます。`p=0.9` に設定すると、確率質量の 90% が上位 2 つのトークンに集中している場合、サンプリング対象として上位 2 つのトークンが絞り込まれます。90% が 10 個のトークンに分散している場合は、同様にサンプリング対象として上位 10 個のトークンが絞り込まれます。

In [ ]:
sampler = keras_hub.samplers.TopPSampler(p=0.5)
output_tokens = sampler(
    next=next,
    prompt=prompt_tokens,
    index=1,
)
txt = tokenizer.detokenize(output_tokens)
print(f"Top-P search generated text: \n{txt}\n")

### テキスト生成用コールバックの使用

これらのユーティリティをコールバックでラップすることもできます。これにより、モデルのエポックごとに予測シーケンスを出力できます。これは、エポックごとにモデルが改善されているかどうかを確認するのに非常に役立ちます。以下は top-k search 用のコールバックの例です:

In [ ]:
class TopKTextGenerator(keras.callbacks.Callback):
    """A callback to generate text from a trained model using top-k."""

    def __init__(self, k):
        self.sampler = keras_hub.samplers.TopKSampler(k)

    def on_epoch_end(self, epoch, logs=None):
        output_tokens = self.sampler(
            next=next,
            prompt=prompt_tokens,
            index=1,
        )
        txt = tokenizer.detokenize(output_tokens)
        print(f"Top-K search generated text: \n{txt}\n")


text_generation_callback = TopKTextGenerator(k=10)
# コールバックを実演するためのダミーのトレーニング ループ。
model.fit(
    train_ds.take(1), verbose=2, epochs=2, callbacks=[text_generation_callback]
)

## 謝辞
このノートブックは、[Jesse Chan による Keras チュートリアル](https://keras.io/examples/generative/text_generation_gpt/#train-the-tokenizer) に基づいています。Transformer デコーダー レイヤは、Google による研究論文 [Attention Is All You Need, Vaswani et al., 2017](https://arxiv.org/abs/1706.03762) に基づいています。

# License

Copyright 2022 Google Inc. Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at http://www.apache.org/licenses/LICENSE-2.0 Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License